# LCEL 빌딩 블록


## OpenAI LLM 준비


In [30]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# .env 파일 로드
load_dotenv()

# OpenAI LLM 준비
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

print("✅ LLM 준비 완료")

✅ LLM 준비 완료


## 프롬프트 메시지 구성


In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 명탐정 코난 영화 전문가입니다. 질문에 대해 명확하고 논리적으로 답하세요.",
        ),
        (
            "human",
            "{question} **{instruction}** 영화 제목과 개봉 연도만 간결하게 답변하세요.",
        ),
    ]
)

# 동일한 프롬프트에서 partial()로 변수를 다르게 바인딩
descending_prompt = prompt.partial(instruction="개봉 연도 내림차순으로 답변하세요.")
ascending_prompt = prompt.partial(instruction="개봉 연도 오름차순으로 답변하세요.")

print("✅ 프롬프트 템플릿 메시지 구성 완료 (partial 바인딩 적용)")

✅ 프롬프트 템플릿 메시지 구성 완료 (partial 바인딩 적용)


## LCEL 체인 구성

### 영화 개봉 연도 내림차순 출력


In [32]:
from langchain_core.output_parsers import StrOutputParser

descending_chain = descending_prompt | llm | StrOutputParser()

descending_movies = descending_chain.invoke(
    {
        "question": "가장 최근에 영화로 개봉된 '명탐정 코난' 한국어 제목과 개봉 연도를 5개 알려주세요."
    }
)
print(f"✅ 명탐정 코난 영화 내림차순 정리: \n{descending_movies}")

✅ 명탐정 코난 영화 내림차순 정리: 
1. 명탐정 코난: 흑철의 어영 - 2023년
2. 명탐정 코난: 할로윈의 신부 - 2022년
3. 명탐정 코난: 비색의 탄환 - 2021년
4. 명탐정 코난: 감청의 권 - 2019년
5. 명탐정 코난: 제로의 집행인 - 2018년


### 영화 개봉 연도 오름차순 출력


In [33]:
ascending_chain = ascending_prompt | llm | StrOutputParser()

ascending_movies = ascending_chain.invoke(
    {
        "question": "가장 최근에 영화로 개봉된 '명탐정 코난' 한국어 제목과 개봉 연도를 5개 알려주세요."
    }
)
print(f"✅ 명탐정 코난 영화 오름차순 정리: \n{ascending_movies}")

✅ 명탐정 코난 영화 오름차순 정리: 
1. 명탐정 코난: 제로의 집행인 - 2018년
2. 명탐정 코난: 감청의 권 - 2019년
3. 명탐정 코난: 비색의 탄환 - 2021년
4. 명탐정 코난: 흑철의 어영 - 2022년
5. 명탐정 코난: 흑철의 어영 - 2023년


### RunnableParallel을 활용한 복합 체인 구성


In [35]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

parallel_chain = RunnableParallel(
    {
        "ascending_movies": ascending_chain,
        "descending_movies": descending_chain,
    }
)

all_movies = RunnableLambda(
    lambda x: f"내림차순:\n{x['descending_movies']}\n오름차순:\n{x['ascending_movies']}"
)
combined_chain = parallel_chain | all_movies | StrOutputParser()
results = combined_chain.invoke(
    {
        "question": "가장 최근에 영화로 개봉된 '명탐정 코난' 한국어 제목과 개봉 연도를 5개 알려주세요."
    }
)

print(f"✅ 병렬 체인 실행 완료:\n{results}")

✅ 병렬 체인 실행 완료:
내림차순:
1. 명탐정 코난: 흑철의 어영 - 2023년
2. 명탐정 코난: 할로윈의 신부 - 2022년
3. 명탐정 코난: 비색의 탄환 - 2021년
4. 명탐정 코난: 감청의 권 - 2019년
5. 명탐정 코난: 제로의 집행인 - 2018년
오름차순:
1. 명탐정 코난: 제로의 집행인 - 2018년
2. 명탐정 코난: 감청의 권 - 2019년
3. 명탐정 코난: 비색의 탄환 - 2021년
4. 명탐정 코난: 할로윈의 신부 - 2022년
5. 명탐정 코난: 흑철의 어영 - 2023년
